# 02 - Data Preprocessing for PyTorch

This notebook prepares the UCI HAR dataset for deep learning.

Objectives:
- Inspect the existing sensor signal distributions.
- Understand the participant-wise train/test split.
- Convert activity labels into zero-based indices.
- Prepare signals for a PyTorch 1D CNN.
- Build PyTorch Dataset and DataLoader objects.

## Imports and load existing data

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from paths import (
    TRAIN_SIGNALS_DIR,
    TEST_SIGNALS_DIR,
    TRAIN_LABELS_FILE,
    TEST_LABELS_FILE,
    TRAIN_SUBJECTS_FILE,
    TEST_SUBJECTS_FILE,
)

CHANNELS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
]

def load_signals(signals_dir, split):
    channel_data = []

    for channel in CHANNELS:
        file_path = signals_dir / f"{channel}_{split}.txt"
        data = np.loadtxt(file_path)
        channel_data.append(data)

    return np.stack(channel_data, axis=-1)


def load_labels(labels_file):
    return np.loadtxt(labels_file, dtype=int)


X_train = load_signals(TRAIN_SIGNALS_DIR, "train")
y_train = load_labels(TRAIN_LABELS_FILE)

X_test = load_signals(TEST_SIGNALS_DIR, "test")
y_test = load_labels(TEST_LABELS_FILE)

print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (7352, 128, 6) (7352,)
Testing: (2947, 128, 6) (2947,)


## Inspect sensor value ranges

In [3]:
print("Training data statistics:\n")

for i, channel in enumerate(CHANNELS):
    values = X_train[:, :, i]

    print(f"{channel}:")
    print(f"  Minimum: {values.min():.4f}")
    print(f"  Maximum: {values.max():.4f}")
    print(f"  Mean:    {values.mean():.4f}")
    print(f"  Std:     {values.std():.4f}")
    print()

Training data statistics:

body_acc_x:
  Minimum: -1.2322
  Maximum: 1.2999
  Mean:    -0.0006
  Std:     0.1948

body_acc_y:
  Minimum: -1.3453
  Maximum: 0.9760
  Mean:    -0.0003
  Std:     0.1224

body_acc_z:
  Minimum: -1.3647
  Maximum: 1.0669
  Mean:    -0.0003
  Std:     0.1069

body_gyro_x:
  Minimum: -4.7337
  Maximum: 4.1555
  Mean:    0.0005
  Std:     0.4068

body_gyro_y:
  Minimum: -5.9743
  Maximum: 5.7461
  Mean:    -0.0008
  Std:     0.3819

body_gyro_z:
  Minimum: -2.7630
  Maximum: 2.3660
  Mean:    0.0001
  Std:     0.2557



## Inspect participant-wise split

In [4]:
train_subjects = np.loadtxt(
    TRAIN_SUBJECTS_FILE,
    dtype=int
)

test_subjects = np.loadtxt(
    TEST_SUBJECTS_FILE,
    dtype=int
)

print("Training participants:", np.unique(train_subjects))
print("Testing participants:", np.unique(test_subjects))

print(
    "Shared participants:",
    np.intersect1d(
        np.unique(train_subjects),
        np.unique(test_subjects)
    )
)

Training participants: [ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
Testing participants: [ 2  4  9 10 12 13 18 20 24]
Shared participants: []


## Convert labels
Our current activity labels are: 1 2 3 4 5 6
Expected PyTorch labels: 0 1 2 3 4 5

In [5]:
# Convert activity labels from 1-6 to 0-5

y_train_encoded = y_train - 1
y_test_encoded = y_test - 1

print("Original training labels:", np.unique(y_train))
print("Encoded training labels:", np.unique(y_train_encoded))

print("Original testing labels:", np.unique(y_test))
print("Encoded testing labels:", np.unique(y_test_encoded))

Original training labels: [1 2 3 4 5 6]
Encoded training labels: [0 1 2 3 4 5]
Original testing labels: [1 2 3 4 5 6]
Encoded testing labels: [0 1 2 3 4 5]


## Verify label distribution

In [6]:
print("Training label distribution:")

unique_labels, counts = np.unique(
    y_train_encoded,
    return_counts=True
)

for label, count in zip(unique_labels, counts):
    print(f"{label}: {count}")

print("\nTesting label distribution:")

unique_labels, counts = np.unique(
    y_test_encoded,
    return_counts=True
)

for label, count in zip(unique_labels, counts):
    print(f"{label}: {count}")

Training label distribution:
0: 1226
1: 1073
2: 986
3: 1286
4: 1374
5: 1407

Testing label distribution:
0: 496
1: 471
2: 420
3: 491
4: 532
5: 537


## Prepare signal dimensions for a 1D CNN

Our original signal shape is:  
(samples, time_steps, channels)  
(7352, 128, 6)  


However, PyTorch's nn.Conv1d expects:  
(batch_size, channels, sequence_length)  


Therefore, we need to rearrange the dimensions:

In [7]:
# Rearrange dimensions for PyTorch Conv1d

X_train_cnn = np.transpose(X_train, (0, 2, 1))
X_test_cnn = np.transpose(X_test, (0, 2, 1))

print("Training CNN input shape:", X_train_cnn.shape)
print("Testing CNN input shape:", X_test_cnn.shape)

Training CNN input shape: (7352, 6, 128)
Testing CNN input shape: (2947, 6, 128)


## Verify the prepared data

In [8]:
print("Training signals:", X_train_cnn.shape)
print("Training labels:", y_train_encoded.shape)

print("Testing signals:", X_test_cnn.shape)
print("Testing labels:", y_test_encoded.shape)

print("\nData validation:")

print("Training signals finite:", np.isfinite(X_train_cnn).all())
print("Testing signals finite:", np.isfinite(X_test_cnn).all())

print("Training labels valid:", np.all(
    (y_train_encoded >= 0) & (y_train_encoded < 6)
))

print("Testing labels valid:", np.all(
    (y_test_encoded >= 0) & (y_test_encoded < 6)
))

Training signals: (7352, 6, 128)
Training labels: (7352,)
Testing signals: (2947, 6, 128)
Testing labels: (2947,)

Data validation:
Training signals finite: True
Testing signals finite: True
Training labels valid: True
Testing labels valid: True


## Convert signals into tensors

In [9]:
# Convert NumPy arrays to contiguous arrays
X_train_np = np.ascontiguousarray(X_train_cnn)
X_test_np = np.ascontiguousarray(X_test_cnn)

# Convert signals to float32 tensors
X_train_tensor = torch.from_numpy(X_train_np).float()
X_test_tensor = torch.from_numpy(X_test_np).float()

print("Training tensor:", X_train_tensor.shape)
print("Testing tensor:", X_test_tensor.shape)

Training tensor: torch.Size([7352, 6, 128])
Testing tensor: torch.Size([2947, 6, 128])


## Convert labels into tensors

In [10]:
# Convert labels to PyTorch long tensors
y_train_tensor = torch.from_numpy(
    y_train_encoded
).long()

y_test_tensor = torch.from_numpy(
    y_test_encoded
).long()

print("Training labels:", y_train_tensor.shape)
print("Testing labels:", y_test_tensor.shape)

Training labels: torch.Size([7352])
Testing labels: torch.Size([2947])


## Inspect tensor properties

In [11]:
print("Training signals:")
print("Shape:", X_train_tensor.shape)
print("Data type:", X_train_tensor.dtype)

print("\nTraining labels:")
print("Shape:", y_train_tensor.shape)
print("Data type:", y_train_tensor.dtype)

print("\nTesting signals:")
print("Shape:", X_test_tensor.shape)
print("Data type:", X_test_tensor.dtype)

print("\nTesting labels:")
print("Shape:", y_test_tensor.shape)
print("Data type:", y_test_tensor.dtype)

Training signals:
Shape: torch.Size([7352, 6, 128])
Data type: torch.float32

Training labels:
Shape: torch.Size([7352])
Data type: torch.int64

Testing signals:
Shape: torch.Size([2947, 6, 128])
Data type: torch.float32

Testing labels:
Shape: torch.Size([2947])
Data type: torch.int64


## Import Dataset and DataLoader

In [12]:
from torch.utils.data import Dataset, DataLoader

## Create the custom Dataset class

In [13]:
class HAR_Dataset(Dataset):

    def __init__(self, signals, labels):
        self.signals = signals
        self.labels = labels

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

## Create training and testing datasets

In [14]:
train_dataset = HAR_Dataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = HAR_Dataset(
    X_test_tensor,
    y_test_tensor
)

## Verify the Dataset

In [15]:
sample_signal, sample_label = train_dataset[0]

print("Signal shape:", sample_signal.shape)
print("Signal data type:", sample_signal.dtype)

print("\nActivity label:", sample_label.item())
print("Label data type:", sample_label.dtype)

print("\nTraining dataset size:", len(train_dataset))
print("Testing dataset size:", len(test_dataset))

Signal shape: torch.Size([6, 128])
Signal data type: torch.float32

Activity label: 4
Label data type: torch.int64

Training dataset size: 7352
Testing dataset size: 2947


## Create DataLoaders

In [16]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

## Verify the batches

In [17]:
signals_batch, labels_batch = next(iter(train_loader))

print("Signal batch shape:", signals_batch.shape)
print("Label batch shape:", labels_batch.shape)

print("\nSignal data type:", signals_batch.dtype)
print("Label data type:", labels_batch.dtype)

print("\nNumber of training batches:", len(train_loader))
print("Number of testing batches:", len(test_loader))

Signal batch shape: torch.Size([64, 6, 128])
Label batch shape: torch.Size([64])

Signal data type: torch.float32
Label data type: torch.int64

Number of training batches: 115
Number of testing batches: 47


## Load the training participant IDs

In [18]:
train_subjects = np.loadtxt(
    TRAIN_SUBJECTS_FILE,
    dtype=int
)

print("Participant IDs shape:", train_subjects.shape)
print("Unique training participants:", np.unique(train_subjects))

Participant IDs shape: (7352,)
Unique training participants: [ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]


## Create a participant-independent validation split

In [19]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_indices, val_indices = next(
    splitter.split(
        X_train_tensor,
        y_train_tensor,
        groups=train_subjects
    )
)

## Check the participant separation

In [20]:
train_participants = np.unique(train_subjects[train_indices])
val_participants = np.unique(train_subjects[val_indices])

print("Training participants:", train_participants)
print("Validation participants:", val_participants)

print(
    "Shared participants:",
    np.intersect1d(train_participants, val_participants)
)

print("\nTraining samples:", len(train_indices))
print("Validation samples:", len(val_indices))

Training participants: [ 5  6  7  8 11 14 16 17 19 21 22 23 26 28 29 30]
Validation participants: [ 1  3 15 25 27]
Shared participants: []

Training samples: 5551
Validation samples: 1801


## Create the three datasets and DataLoaders

In [21]:
from torch.utils.data import Subset

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

## Verify all three DataLoaders

In [22]:
print("Dataset sizes:")
print("Training:", len(train_subset))
print("Validation:", len(val_subset))
print("Testing:", len(test_dataset))

print("\nNumber of batches:")
print("Training:", len(train_loader))
print("Validation:", len(val_loader))
print("Testing:", len(test_loader))

print("\nBatch shape verification:")

for name, loader in [
    ("Training", train_loader),
    ("Validation", val_loader),
    ("Testing", test_loader)
]:
    signals, labels = next(iter(loader))

    print(f"\n{name}:")
    print("Signals:", signals.shape)
    print("Labels:", labels.shape)

Dataset sizes:
Training: 5551
Validation: 1801
Testing: 2947

Number of batches:
Training: 87
Validation: 29
Testing: 47

Batch shape verification:

Training:
Signals: torch.Size([64, 6, 128])
Labels: torch.Size([64])

Validation:
Signals: torch.Size([64, 6, 128])
Labels: torch.Size([64])

Testing:
Signals: torch.Size([64, 6, 128])
Labels: torch.Size([64])


## final quality check

In [23]:
activity_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

def show_class_distribution(name, labels):
    counts = np.bincount(
        labels.numpy(),
        minlength=6
    )

    print(f"\n{name}")
    for i, count in enumerate(counts):
        print(f"{activity_names[i]}: {count}")

show_class_distribution(
    "Training",
    y_train_tensor[train_indices]
)

show_class_distribution(
    "Validation",
    y_train_tensor[val_indices]
)

show_class_distribution(
    "Testing",
    y_test_tensor
)


Training
WALKING: 888
WALKING_UPSTAIRS: 797
WALKING_DOWNSTAIRS: 744
SITTING: 993
STANDING: 1053
LAYING: 1076

Validation
WALKING: 338
WALKING_UPSTAIRS: 276
WALKING_DOWNSTAIRS: 242
SITTING: 293
STANDING: 321
LAYING: 331

Testing
WALKING: 496
WALKING_UPSTAIRS: 471
WALKING_DOWNSTAIRS: 420
SITTING: 491
STANDING: 532
LAYING: 537


In [24]:
print(
    "Training-validation overlap:",
    np.intersect1d(
        train_subjects[train_indices],
        train_subjects[val_indices]
    )
)

print(
    "Training-testing overlap:",
    np.intersect1d(
        train_subjects[train_indices],
        np.loadtxt(TEST_SUBJECTS_FILE, dtype=int)
    )
)

print(
    "Validation-testing overlap:",
    np.intersect1d(
        train_subjects[val_indices],
        np.loadtxt(TEST_SUBJECTS_FILE, dtype=int)
    )
)

Training-validation overlap: []
Training-testing overlap: []
Validation-testing overlap: []


## Save prepared datasets

Save the processed tensors and dataset split indices so they can
be reused in the model development notebook.

In [25]:

import torch

# Define the output file
processed_data_path = PROJECT_ROOT / "data" / "processed"
processed_data_path.mkdir(parents=True, exist_ok=True)

output_file = processed_data_path / "har_preprocessed.pt"

# Save tensors and split indices
torch.save(
    {
        "X_train": X_train_tensor,
        "X_test": X_test_tensor,
        "y_train": y_train_tensor,
        "y_test": y_test_tensor,
        "train_indices": train_indices,
        "val_indices": val_indices,
    },
    output_file
)

print("Prepared data saved to:", output_file)
print("File exists:", output_file.exists())


Prepared data saved to: d:\sensor-har-pytorch-accelerators\data\processed\har_preprocessed.pt
File exists: True
